In [1]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import SGDClassifier
from lightgbm import LGBMClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
import scipy

In [12]:
cv_res

{'fit_time': array([   8.2230525 ,  994.27887774, 1160.53301358, 1124.2809546 ,
        1160.78306532]),
 'score_time': array([0.        , 1.07693744, 0.89871764, 0.9818666 , 0.86640787]),
 'test_accuracy': array([       nan, 0.77837838, 0.72972973, 0.74054054, 0.73369565]),
 'test_precision': array([       nan, 0.64285714, 0.48275862, 0.52380952, 0.47368421]),
 'test_recall': array([       nan, 0.36734694, 0.28571429, 0.2244898 , 0.1875    ]),
 'test_f1': array([       nan, 0.46753247, 0.35897436, 0.31428571, 0.26865672]),
 'test_roc_auc': array([       nan, 0.75960384, 0.74129652, 0.70258103, 0.70128676])}

In [4]:
dataset_druggable = pd.read_csv("../input/combined_DepMap_21Q3_druggable.csv")
dataset_ccle = pd.read_csv("../input/combined_DepMap_21Q3_CCLE_expression.csv")
druggable_X = dataset_druggable.iloc[:, 1:-4686]
druggable_y = dataset_druggable.iloc[:, -4686:]
ccle_X = dataset_ccle.iloc[:, 1:-4686]
ccle = dataset_ccle.iloc[:, -4686:]


In [2]:
dataset = pd.read_csv("./input/combined_DepMap_21Q3.csv")
num_gene = 17651
drug_X = dataset.iloc[:, 1:num_gene+1]
drug_y = dataset.iloc[:, -4686:]
drug_list = drug_y.columns.tolist()

In [15]:
import time

In [3]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

In [ ]:
xgb1 = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 1 )


In [11]:
xgb = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 20 )

In [12]:
X_train, X_test, y_train, y_test = train_test_split(drug_X, drug_y['BRD-A00077618-236-07-6::2.5::HTS'], test_size=0.2, random_state=20)

In [13]:
%timeit cross_validate(xgb, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=1)

1min 44s ± 5.2 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:


%timeit cross_validate(xgb1, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=20)


1min 46s ± 1.5 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
xgb2 = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 4 )
%timeit cross_validate(xgb2, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=5)


48.6 s ± 1.92 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
xgb3 = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 15 )
%timeit cross_validate(xgb3, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=5)


28.7 s ± 673 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
from sklearn.metrics import get_scorer
for s in ['balanced_accuracy', 'precision', 'recall', 'f1', 'average_precision', 'roc_auc']:
    score_ = get_scorer(s)
    print(score_(xgb3, X_test, y_test))

0.5384615384615384
1.0
0.07692307692307693
0.14285714285714285
0.5404300447624246
0.7521952932911837


In [ ]:
xgb3.fit(X_train, y_train)

In [ ]:
score_(xgb3, X_test, y_test)

np.float64(0.5404300447624246)

In [ ]:
# SOME PARAMETERS FOR GRIDSEARCH
GRID_SEARCH_PARAM = {
'XGB':{
 	"learning_rate" : [0.05,0.10,0.15,0.20],
 	"max_depth" : [ 3, 4, 5, 6, 8, 10, 12, 15],
 	"min_child_weight" : [ 1, 3, 5, 7 ],
 	"gamma": [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
	 "n_estimators": [50, 100, 200]
},
 'RF':{
	 "n_estimators": [100, 150, 250],
	 "max_depth" : [20, 50, 100, 200],
 },
 'SGD':{
	 'l1_ratio':[0.2, 0.15, 0.1, 0.05],
	 'alpha':[0.02, 0.05]
 }
}

# Basic function to handel sklearn and traditional model training and basic hyperparameter optimization
# TODO refactor this into a class maybe, class DrugModel
def skl_drug_model(X, Y, drug, model = XGBClassifier, oversample = True, fixed_params={}, search_params = {}, search_method = 'gridcv'):

	assert model in [XGBClassifier, RandomForestClassifier, SGDClassifier, LGBMClassifier], f' {model} type not supported'

	
	y = Y[drug]

	 # split X and y into training and testing sets
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


	if oversample:
		ros = SMOTE(random_state=72)
		X_res, y_res = ros.fit_resample(X_train, y_train)
		oversample = 'oversample'
	else:
		X_res, y_res = X_train, y_train
		oversample = ''


	 # instantiate the classifier
	 

	# k-fold cross validation using multiple metric evaluation
	kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
	 #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
	if oversample:
		imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
							  ('classifier', model(n_jobs=10, **fixed_params))])
		grid_search_parameters = {'classifier__' + key: search_params[key] for key in search_params}
	else:
		imba_pipeline = model(**fixed_params, n_jobs=20)
		grid_search_parameters = search_params
	 #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
	model0 = model(**fixed_params, n_jobs=20)

	 #perform gridsearch if needed
	if search_method == 'gridcv' and search_params:
		grid_imba = HalvingRandomSearchCV(imba_pipeline, param_distributions=grid_search_parameters, cv=kfold, scoring='precision')
		#grid_imba.fit(X_train, y_train) 
		cv_results = cross_validate(grid_imba, X, y, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=10)
		print(cv_results)
		print(grid_imba)
		#return cv_results, grid_imba
		#best_params = {key.removeprefix('classifier__'):grid_imba.best_params_[key] for key in grid_imba.best_params_}
		#print(best_params)
		#model0.set_params(**best_params)
	else:
		cv_results = cross_validate(model0, X_train, y_train, scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold, njobs = 10)
		grid_imba = model0
	 
	if oversample:
		imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
							  ('classifier', model0)])
	else:
		imba_pipeline = model0
		  
	# generate cross validation results of best model with correct oversampling 
	# OVERSAMPLING must come after validation split for correct validation , thus the use of pipeline
	# cross_validate function will first split into train/validate, then feed training data into pipeline (oversampling + training)
	#cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 20)
	cv_results = pd.DataFrame(cv_results)
	print(cv_results)

	 # declare parameters
	 

	 # fit the classifier to the training data
	model0.fit(X_res, y_res)

	 # save the trained model
	
		  
	final_results = {
		'best_model': model0,
		'drug': drug,
		'model_class': model,
		'model_search': grid_imba,
		'search_method': search_method,
		'X_test': X_test,
		'Y_test': y_test,
		'X_train': X_train,
		'Y_train': y_train,
		'cv_results':cv_results,
		'oversample': True if oversample else False
	}

	return final_results


def write_drug_model_result(model_results, out_dir):
	 
	model0 = model_results['best_model']
	oversample = 'oversample' if model_results['oversample'] else ''
	drug = model_results['drug']
	model_class = model_results['model_class']

	model_name = {
		  XGBClassifier:'XGBClassifier',
		  RandomForestClassifier:'RandomForestClassifier',
		  SGDClassifier:'SGDClassifier',
		  LGBMClassifier:'LGBMClassifier'

	}[model_class]

	full_out_dir = f'{out_dir}/{oversample}/{drug}/'
	X_test, y_test = model_results['X_test'], model_results['Y_test']
	X_train, y_train = model_results['X_train'], model_results['Y_train']
	y_pred = model0.predict(X_test)

	os.makedirs(full_out_dir, exist_ok=True)

	joblib.dump(model0, f'{full_out_dir}/{model_name}_{drug}.joblib')
	
	model_results['cv_results'].to_csv(f'{full_out_dir}/{model_name}_cv_results_{drug}.csv')

	print(drug,
		 f'{model_class}_model_parameters', model0, "\n",
		 "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
		 file=open(f'{full_out_dir}/{model_name}_confusion_matrix.txt', "a"))

	model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
	model_report = pd.DataFrame(model_report).transpose()
	
	if (model_report.index == "1").any() == True:
		r1 = pd.DataFrame(model_report.loc["1"]).transpose()
		r1.to_csv(f'{full_out_dir}/{model_name}_classification_report_{drug}.csv')
	else:
		print(drug, "Nothing predicted as 1",
			   file=open(f'{out_dir}/{oversample}/{model_name}_classification_report_log.txt', "a"))

	 

	 # feature importance with XGBoost
	if model_class in [XGBClassifier, RandomForestClassifier]:
		fi = pd.DataFrame({'feature': list(X_train.columns),
					'importances': model0.feature_importances_ * 100}).\
					 sort_values('importances', ascending = False)
		fi.to_csv(f'{full_out_dir}/{model_name}_feature_importance_{drug}.csv')

		# plot feature importance
		plt.figure(figsize=(10, 8))
		sns.barplot(x='importances', y='feature', data=fi.head(20))
		plt.title(f'{model_name} Feature Importance for {drug}')
		plt.tight_layout()
		plt.savefig(f'{full_out_dir}/{model_name}_feature_importance_{drug}.png')
		plt.close()
	# save the model report
	model_report.to_csv(f'{full_out_dir}/{model_name}_classification_report_{drug}.csv')
	print(f"Model for {drug} saved to {full_out_dir}/{model_name}_{drug}.joblib")
	return model_results




In [15]:
model_map = {}
for d in drug_list[0:1]:
	cv_res1, grid_imba1 = skl_drug_model(drug_X, drug_y, d, 
		model = XGBClassifier, 
		oversample = False, 
		fixed_params={},
		search_params=GRID_SEARCH_PARAM['XGB']
	)


In [ ]:
model_map = {}
for d in drug_list[0:1]:
	model_map[d] = skl_drug_model(drug_X, drug_y, d, 
		model = LGBMClassifier, 
		oversample = False,
		search_params={},
		fixed_params={}
		)

In [ ]:
write_drug_model_result(model_map['BRD-A00077618-236-07-6::2.5::HTS'], out_dir = 'output')

In [ ]:
starttime = time.time()
grid_search = skl_drug_model('BRD-A00100033-001-08-9::2.5::HTS',   oversample = True, model = RandomForestClassifier
)
endtime = time.time()

{'max_depth': 200, 'n_estimators': 100}
